In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [2]:
# 加载模型与TOkenizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from unsloth import FastLanguageModel

max_length = 2048 # Supports automatic RoPE Scaling, so choose any number
model_name = "Qwen/Qwen3-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_length,
    dtype=None,  # For auto-detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
    load_in_4bit=True,  # Use 4bit quantization to reduce memory usage. Can be False
)

# Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Dropout = 0 is currently optimized
    bias="none",  # Bias = "none" is currently optimized
    use_gradient_checkpointing=True,
    random_state=3407,
)


==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090 Ti. Num GPUs = 1. Max memory: 23.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:02<00:00, 184.97it/s]


unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [3]:
# 数据集 处理数据集至openai格式
from datasets import load_dataset
dataset_dict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl",
                                              "test":"data/keywords_data_test.jsonl"})
# 转成openai格式
def map_func(exapmle):
    conversation = exapmle["conversation"]
    messages=[]
    for item in conversation:
        messages.append({"role":"user","content":item["human"]})
        messages.append({"role":"assistant","content":item["assistant"]})
    return {"messages":messages}

dataset_dict=dataset_dict.map(map_func,batched=False,remove_columns=["conversation_id","category","conversation","dataset"])

Generating train split: 49500 examples [00:00, 131383.70 examples/s]
Generating test split: 500 examples [00:00, 152177.06 examples/s]
Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 500/500 [00:00<00:00, 24971.45 examples/s]


In [6]:
from unsloth.chat_templates import CHAT_TEMPLATES
print(list(CHAT_TEMPLATES.keys()))

['unsloth', 'zephyr', 'chatml', 'mistral', 'llama', 'vicuna', 'vicuna_old', 'vicuna old', 'alpaca', 'gemma', 'gemma_chatml', 'gemma2', 'gemma2_chatml', 'llama-3', 'llama3', 'phi-3', 'phi-35', 'phi-3.5', 'llama-3.1', 'llama-31', 'llama-3.2', 'llama-3.3', 'llama-32', 'llama-33', 'qwen-2.5', 'qwen-25', 'qwen25', 'qwen2.5', 'phi-4', 'gemma-3', 'gemma3', 'qwen-3', 'qwen3', 'gemma-3n', 'gemma3n', 'gemma-4', 'gemma4', 'gemma-4-thinking', 'gemma4-thinking', 'gpt-oss', 'gptoss', 'qwen3-instruct', 'qwen3-thinking', 'lfm-2', 'lfm-2.5', 'starling', 'yi-chat']


In [13]:

from trl import SFTConfig,SFTTrainer
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3"
)

# def formatting_prompts_func(examples):
#    convos = examples["messages"]
#    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
#    return  texts
def formatting_prompts_func(examples):
    # 情况 A: 如果 examples 是一个 Batch (含有 "messages" 键且值是 nested list)
    if isinstance(examples["messages"], list) and len(examples["messages"]) > 0 and isinstance(examples["messages"][0], list):
        convos = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
        return texts
    
    # 情况 B: 如果 examples 是单条数据 (next(iter(dataset)) 的情况)
    else:
        convo = examples["messages"]
        # 直接对这一条对话应用模板
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        return [text] # 必须返回 list



training_args = SFTConfig(
    output_dir="/home/tianjp/llmLearn/stf/Qwen3-4B/sft-unsloth",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_total_limit=2,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    assistant_only_loss=True,
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    #### 此处必须定义 ####
    formatting_func=formatting_prompts_func
    
)



In [14]:
dataloader = trainer.get_train_dataloader()
batch=next(iter(dataloader))
batch["input_ids"].shape

torch.Size([4, 213])

In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49,500 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,2.281992,2.211669
200,2.185795,2.108172
300,2.197684,2.095116
400,2.196504,2.087615
500,2.130648,2.081624
600,2.150870,2.077979
700,2.044657,2.074253
800,2.029052,2.071814
900,2.091783,2.069797
1000,2.141077,2.069268


/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and

TrainOutput(global_step=1000, training_loss=2.209294172286987, metrics={'train_runtime': 2205.2302, 'train_samples_per_second': 3.628, 'train_steps_per_second': 0.453, 'total_flos': 4.60778743044096e+16, 'train_loss': 2.209294172286987, 'epoch': 0.16161616161616163})

In [16]:
trainer.save_model("/home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/best")

/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in unsloth/qwen3-4b-unsloth-bnb-4bit.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in unsloth/qwen3-4b-unsloth-bnb-4bit - will assume that the vocabulary was not modified.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in /home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/best/tokenizer_config.json.


In [ ]:
model.save_pretrained_merged("/home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/merged", tokenizer, save_method = "merged_16bit",)

Found HuggingFace hub cache directory: /home/tianjp/.cache/huggingface/hub


Returning existing local_dir `/home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/merged` as remote repo cannot be accessed in `snapshot_download` ([SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)).
[huggingface_hub._snapshot_download|WARNING]Returning existing local_dir `/home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/merged` as remote repo cannot be accessed in `snapshot_download` ([SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)).


Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|████████████████████████████████████████████████████████████████| 2/2 [03:48<00:00, 114.06s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████████████████████████████████████████████████████████████████| 2/2 [04:07<00:00, 123.98s/it]


Unsloth: Merge process complete. Saved to `/home/tianjp/llmLearn/stf/Qwen3-4B/unsloth/merged`
